In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Fingers on the CPU: MediaPipe Hands

YOLO-pose sees your arm; it does not see your fingers. The pinch that
drives the gripper, the hand size that drives forward/back reach, and the
thumb-to-index line that drives wrist roll all come from MediaPipe Hands,
and it runs on the CPU.

That placement is a *decision*, and deciding where models run is half of
what this workshop is about:

| compute | tenant | why |
|---|---|---|
| NPU | 2x YOLOv26s | camera-rate vision, latency-critical, fits the context budget |
| iGPU | Llama 3.2 3B | 3B parameters; nothing else has the FLOPs |
| **CPU** | **MediaPipe Hands** | small and quick enough for frame rate, ideal for CPU |

**Matching TODO:** `1.4` in `project/models/hands.py`, the initialization.
The landmark-to-signal geometry is provided (it's generic math you can read
in `HandTracker.process`).

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import time

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.quiet import quiet_stderr  # OS-level hush for C++ chatter
from models.hands import HandTracker

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# Inline image display helper (BGR -> notebook).


def show(img_bgr, title="", figsize=(9, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Initializing MediaPipe on the CPU

MediaPipe defaults to CPU inference on Linux, so here we simply
*initialize*: no EPs, no provider_options. The values come from the
`mediapipe:` section of the workshop config, and two of them matter for
robotics use:

- **`static_image_mode=False`** (video mode): the tracker *tracks* between
  frames instead of re-detecting from scratch, which is much cheaper and
  smoother.
- **`max_num_hands: 1`**: one operator, one hand, no ambiguity about who
  drives the gripper.

This is the exact shape of TODO 1.4:

In [ ]:
with quiet_stderr():
    import mediapipe as mp

    m = cfg["mediapipe"]
    hands = mp.solutions.hands.Hands(
        static_image_mode=False,  # video mode: track, don't re-detect
        max_num_hands=int(m["max_num_hands"]),
        min_detection_confidence=float(m["min_detection_confidence"]),
        min_tracking_confidence=float(m["min_tracking_confidence"]),
    )
    # one warmup frame so the graph's init-time log lines land here, silenced
    hands.process(np.zeros((64, 64, 3), dtype=np.uint8))
print("MediaPipe Hands ready (CPU).")

## 2. The signals a hand gives us

The provided `HandTracker` (in `models/hands.py`) turns 21 landmarks into
exactly the signals the mimic mapper consumes:

| signal | geometry | drives |
|---|---|---|
| `pinch` | thumb-tip to index-tip distance / hand size | **gripper** (distance-invariant: 0 = touching) |
| `hand_size` | wrist to middle-knuckle span / frame height | **forward/back reach** (grows as the hand nears the camera) |
| `roll_ti` | *directed* thumb-to-index vector angle | **wrist_roll** (primary cue) |
| `roll_ref` | knuckle-line angle (index to pinky MCP) | wrist_roll steady reference |
| `roll_ti_weight` | thumb/index pixel separation | how much to trust `roll_ti` when pinched shut |

Note the design detail in `roll_ti`: it's a **directed** vector
(thumb to index, full +/-180 degrees), because a folded +/-90 degree angle
sign-flips when your hand tilts past vertical. The `roll_ti_weight` exists
because a pinched-shut hand gives a line too short to accurately determine
a gripper closed position.

Let's watch the signals live. The capture below shows the camera (with
the hand landmarks drawn on it) next to a live strip chart of `pinch`,
`hand_size`, and `roll_ti`, both updating in place while the loop reads
the mount camera. Hold your hand up, pinch a few times - watch the pinch
trace drop toward 0 exactly when the gripper would close - roll your
wrist, then press **stop** and read the full-resolution plots below.

In [ ]:
# HandTracker holds the provided landmark->signal geometry; we hand it the
# mp Hands we just built in section 1, so this cell runs even before TODO 1.4.


def open_mount_camera():
    """The real mount camera: the configured V4L device, else the default webcam."""
    import os

    device = cfg["cameras"]["mount"].get("device", 0)
    candidates = [device, 0]
    if isinstance(device, str) and not pathlib.Path(device).exists():
        candidates = [0]  # configured path absent: go straight to
        # the default webcam (probing a missing
        # path makes OpenCV warn noisily)
    # The AMD ISP node can still be busy for a moment after the previous
    # notebook's kernel releases it. Retry before declaring it stuck.
    for _ in range(8):
        for candidate in candidates:
            cap = (
                cv2.VideoCapture(candidate, cv2.CAP_V4L2)  # V4L2 direct: no
                if isinstance(candidate, str)  # CV_IMAGES probing
                else cv2.VideoCapture(candidate)
            )
            if cap.isOpened():
                print(f"camera open: {candidate}")
                return cap
            cap.release()
        time.sleep(0.25)
    holders = []
    if isinstance(device, str):
        for pid in os.listdir("/proc"):
            if not pid.isdigit():
                continue
            fd_dir = f"/proc/{pid}/fd"
            try:
                fds = os.listdir(fd_dir)
            except OSError:
                continue
            for fd in fds:
                try:
                    target = os.readlink(f"{fd_dir}/{fd}")
                except OSError:
                    continue
                if target != device:
                    continue
                try:
                    cmd = (
                        open(f"/proc/{pid}/cmdline", "rb")
                        .read()
                        .replace(b"\0", b" ")
                        .decode()
                        .strip()
                    )
                except OSError:
                    cmd = ""
                holders.append(f"pid {pid} ({cmd[:140]})")
                break
    held = ("Still held by:\n  " + "\n  ".join(holders) + "\n") if holders else ""
    raise RuntimeError(
        "could not open the mount camera.\n"
        f"  config expects {device}\n"
        f"{held}"
        "Close the other notebook. Its kernel shuts down on close and "
        "releases the camera. Re-run this cell. Restarting "
        "./run_notebooks.sh is not required."
    )


def to_jpeg(img):
    ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return jpg.tobytes()


# Live strip chart, drawn with cv2 so it keeps up at camera rate (a
# matplotlib redraw per frame would drop the loop to a slideshow).
TRACES = [
    ("pinch -> gripper", (80, 220, 80), (0.0, 2.0)),
    ("hand_size -> fwd/back", (80, 180, 255), (0.0, 0.6)),
    ("roll_ti (deg) -> wrist_roll", (0, 200, 255), (-180.0, 180.0)),
]


def draw_signal_panel(series, width=380, height=480, window=240):
    panel = np.full((height, width, 3), 24, dtype=np.uint8)
    data = series[-window:]
    rows = len(TRACES)
    for i, (name, color, (lo, hi)) in enumerate(TRACES):
        y0 = int(i * height / rows) + 26
        y1 = int((i + 1) * height / rows) - 12
        cv2.putText(
            panel,
            name,
            (10, y0 - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (170, 170, 170),
            1,
        )
        cv2.rectangle(panel, (8, y0), (width - 8, y1), (60, 60, 60), 1)
        if len(data) >= 2:
            pts = []
            for k, sample in enumerate(data):
                v = max(lo, min(hi, float(sample[i])))
                x = 8 + int(k * (width - 16) / max(window - 1, 1))
                pts.append((x, y1 - int((v - lo) / (hi - lo) * (y1 - y0))))
            cv2.polylines(panel, [np.array(pts, np.int32)], False, color, 2)
            cv2.putText(
                panel,
                f"{float(data[-1][i]):+.2f}",
                (width - 86, y0 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                color,
                1,
            )
    return panel


import gc

cap = open_mount_camera()
try:
    tracker = HandTracker(cfg, hands=hands)

    stop_btn = widgets.Button(description="stop", button_style="danger")
    stopped = {"flag": False}
    stop_btn.on_click(lambda _: stopped.update(flag=True))

    # Two persistent Image widgets updated in place (flicker-free): the live
    # camera with the hand overlay, and the live signal chart next to it.
    cam_view = widgets.Image(format="jpeg", width=460)
    sig_view = widgets.Image(format="jpeg", width=290)
    sig_view.value = to_jpeg(draw_signal_panel([]))
    display(widgets.VBox([stop_btn, widgets.HBox([cam_view, sig_view])]))

    series, last_frame, last_obs, last_raw = [], None, None, None
    with ui_events() as poll:
        while not stopped["flag"]:
            poll(10)
            ok, frame = cap.read()
            if not ok:
                print("camera stopped mid-capture")
                break
            with quiet_stderr():  # its first detection logs a
                obs, raw = tracker.process(frame)  # projection note - keep it quiet
            vis = frame
            if obs is not None:
                series.append((obs.pinch, obs.hand_size, obs.roll_ti))
                last_frame, last_obs, last_raw = frame, obs, raw
                if hasattr(tracker, "draw"):
                    vis = tracker.draw(frame.copy(), obs, raw)
            cam_view.value = to_jpeg(vis)
            sig_view.value = to_jpeg(draw_signal_panel(series))
    print(f"collected {len(series)} hand observations")
finally:
    # release() returns while this kernel still owns /dev/video0. The AMD ISP
    # node stays locked until the VideoCapture object is destroyed.
    cap.release()
    cap = None
    gc.collect()

In [ ]:
if series:
    arr = np.array(series)
    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    for ax, i, (name, drives) in zip(
        axes,
        range(3),
        [
            ("pinch", "gripper"),
            ("hand_size", "fwd/back reach"),
            ("roll_ti (deg)", "wrist_roll"),
        ],
    ):
        ax.plot(arr[:, i])
        ax.set_ylabel(name)
        ax.grid(alpha=0.3)
        ax.set_title(f"{name} -> {drives}", fontsize=10, loc="left")
    axes[-1].set_xlabel("frame")
    plt.tight_layout()
    plt.show()
else:
    print(
        "no hand seen - hold your hand up in front of the camera and re-run the capture cell"
    )

## 3. How the signals become joints (a preview of component 3)

In the mimic mapper (provided, `common.motion.MimicMapper`, gains from the
real rig):

- `pinch` normalizes against calibrated `grip_min`/`grip_max` and ramps the
  gripper from the closed floor (43 on this rig's calibration - lower jams
  the jaws into the mechanical stop) up to 90% open;
- `hand_size` compares against an auto-calibrated neutral, so a +/-30% size
  change is full forward/back reach (`shoulder_lift` + `elbow_flex`);
- `roll_ti` (weighted against `roll_ref`) is auto-centered on first sight,
  meaning *however you hold your hand becomes roll zero*, and drives
  `wrist_roll`.

You'll wire these into the arm in notebook `03_integration/01_behavior`.

## 4. Your turn

Open **`project/models/hands.py`** and implement **TODO 1.4**
(`create_hand_tracker`): it's the section 1 cell, reading the config
instead of hardcoding.

Then, from the `project/` directory:

```bash
python -m models.selftest

# to see all the TODOs you have left
python app.py
```

Next notebook: `03_igpu_llama.ipynb`, the brain.